# UEFI Boot Process (Modern Linux Systems)
This notebook focuses on the modern UEFI boot process and how it differs from the legacy BIOS/MBR approach.

## 1. Why was BIOS replaced?

Legacy BIOS was designed in the 1980s and has several limitations:

| BIOS Limitation | UEFI Improvement |
|---|---|
| Supports only MBR by default | Supports GPT (very large disks) |
| Maximum ~2 TB boot disk (with MBR) | Supports disks far larger than 2 TB |
| Limited boot code (446 bytes in MBR) | Boot programs are normal `.efi` files |
| Text interface | Can provide graphical interface |
| Difficult to extend | Modular and extensible |
| No Secure Boot | Supports Secure Boot |

**Analogy**

BIOS is like an old flip phone with a few built-in features.

UEFI is like a modern smartphone that can run many applications.


## 2. What is UEFI?

UEFI stands for **Unified Extensible Firmware Interface**.

It is firmware that initializes hardware and then launches an EFI application from a special partition called the **EFI System Partition (ESP)**.

Unlike BIOS, UEFI does **not** execute code from the MBR.


## 3. GPT and the EFI System Partition

Modern systems usually use:

- GPT (GUID Partition Table)
- EFI System Partition (ESP)

Typical layout:

```
Disk
├── EFI System Partition (FAT32)
│      ├── EFI/
│      │    ├── ubuntu/
│      │    │      grubx64.efi
│      │    ├── Microsoft/
│      │    │      bootmgfw.efi
│      │    └── Boot/
├── Linux partition (ext4)
├── Home
└── Swap
```

The ESP is usually mounted as:

```
/boot/efi
```


## 4. Does UEFI still use GRUB?

**Yes—very often.**

People sometimes think UEFI replaces GRUB. It does **not**.

Instead:

```
UEFI Firmware
      ↓
grubx64.efi
      ↓
Linux kernel
      ↓
systemd/init
```

GRUB is simply compiled as an **EFI application**.

On Ubuntu this file is commonly:

```
/boot/efi/EFI/ubuntu/grubx64.efi
```


## 5. Can Linux boot without GRUB?

Yes.

Some distributions can boot using:

- systemd-boot
- EFISTUB (kernel directly as an EFI executable)
- rEFInd

GRUB is the most feature-rich and the most common general-purpose boot loader, especially for dual-boot systems.


## 6. How does UEFI choose what to boot?

UEFI stores boot entries in firmware NVRAM.

Example:

```
Boot0000  Windows Boot Manager
Boot0001  Ubuntu
Boot0002  USB Drive
```

Linux command:

```bash
sudo efibootmgr
```

shows these entries.

You can also change the boot order with `efibootmgr`.


## 7. Complete Linux UEFI Boot Sequence

```
Power ON
      ↓
UEFI Firmware
      ↓
Hardware initialization
      ↓
Read NVRAM boot entries
      ↓
Load grubx64.efi
      ↓
GRUB menu
      ↓
Load Linux kernel
      ↓
Load initramfs
      ↓
Kernel starts
      ↓
systemd
      ↓
Login screen
```


## 8. What happens in dual boot?

The EFI partition typically contains **both** boot loaders.

```
EFI/
   ubuntu/
       grubx64.efi

   Microsoft/
       bootmgfw.efi
```

Normally GRUB automatically detects Windows and provides a menu.

Even if Windows updates change the default boot entry, GRUB is usually still present; you can restore it by changing the firmware boot order or reinstalling GRUB if necessary.


## 9. Recovery

If Linux does not boot:

1. Boot from a Linux Live USB.
2. Mount the Linux partition.
3. Mount the EFI System Partition.
4. Chroot (optional).
5. Reinstall GRUB:

```bash
sudo grub-install
sudo update-grub
```

Usually this repairs the boot loader **without reinstalling Linux or deleting personal files**.


# BIOS vs UEFI Summary

| BIOS | UEFI |
|---|---|
| Executes MBR code | Executes `.efi` programs |
| MBR | GPT |
| 2 TB limit | Very large disks |
| 446-byte boot code | Normal executable files |
| Older standard | Modern standard |
| Limited features | Secure Boot, GUI, networking, extensible |


# Questions

1. Why is UEFI considered more flexible than BIOS?
2. Does UEFI replace GRUB?
3. Where is `grubx64.efi` usually stored?
4. What is the EFI System Partition?
5. Can Linux boot without GRUB?
6. What command lists UEFI boot entries?
7. Why is GPT preferred over MBR?


# Answers

1. Because it uses normal EFI applications, GPT, Secure Boot, NVRAM boot entries, and is extensible.
2. No. UEFI firmware usually launches an EFI boot loader such as GRUB.
3. Usually in `/boot/efi/EFI/<distribution>/grubx64.efi`.
4. A small FAT32 partition containing EFI boot programs.
5. Yes. For example, with systemd-boot or EFISTUB.
6. `efibootmgr`.
7. GPT supports many more partitions, better reliability, and very large disks.
